# nanoGPT: pre-train on CodeParrot, fine-tune a classifier on xCodeEval (compiles / compile error)
Edit **only the config cell** between runs.

In [1]:
# ---------------- CONFIG ----------------
import torch
POS_ENC = "rope"           # "abs" = GPT-2 learned absolute PE, "rope" = DeepSeek rotary, "nope" = none
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

# Pre-training
PRETRAIN_FILES = 5000      # CodeParrot files tokenized into train.bin
PRETRAIN_ITERS = 3000      # training iterations
BATCH_SIZE, BLOCK_SIZE, GRAD_ACCUM = 12, 256, 1
N_LAYER, N_HEAD, N_EMBD = 4, 4, 128

# xCodeEval labelled data (built once, see "Build labels")
XC_LANG_CLUSTER = "Python"
XC_N_PER_CLASS  = 5000     # examples collected per class
XC_MAX_SHARDS   = 8
LABEL_FILE      = "labels_xcodeeval.json"

# Fine-tuning (classifier)
HEAD_DEPTH  = 1            # 1, 2 or 3 linear layers
HEAD_HIDDEN = 256          # hidden width (ignored when HEAD_DEPTH=1)
N_LABELED   = 4800         # labelled training examples used (None = all), class-balanced
FT_EPOCHS   = 10
N_FREEZE    = 0            # bottom transformer blocks to freeze: 0 = full fine-tune, N_LAYER = fully frozen

PRETRAIN_DIR = f"out-code-{POS_ENC}-{PRETRAIN_ITERS}it-{PRETRAIN_FILES}f"
tok_per_iter = BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM
print(f"{tok_per_iter:,} tokens/iter -> {tok_per_iter*PRETRAIN_ITERS/1e6:.1f}M tokens seen in pre-training")

3,072 tokens/iter -> 9.2M tokens seen in pre-training


## 1. Pre-training data (CodeParrot)

In [2]:
import os, numpy as np, tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
out = "data/codeparrot"                 # relative to the nanoGPT folder
os.makedirs(out, exist_ok=True)

def dump(name, split, max_files):
    ds = load_dataset(name, split=split, streaming=True)
    ids = []
    for i, ex in enumerate(ds):
        if i >= max_files: break
        ids.extend(enc.encode_ordinary(ex["content"]))
        ids.append(enc.eot_token)
    return np.array(ids, dtype=np.uint16)

train = dump("codeparrot/codeparrot-clean", "train", PRETRAIN_FILES)
val   = dump("codeparrot/codeparrot-clean-valid", "train", 500)
train.tofile(os.path.join(out, "train.bin"))
val.tofile(os.path.join(out, "val.bin"))
print(f"{len(train):,} train tokens, {len(val):,} val tokens")
print(f"pre-training will make {tok_per_iter*PRETRAIN_ITERS/len(train):.2f} passes over the data")

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


22,682,655 train tokens, 2,370,285 val tokens
pre-training will make 0.41 passes over the data


## 2. Pre-train

In [4]:
%run train.py --dataset=codeparrot --out_dir={PRETRAIN_DIR} \
  --device={DEVICE} --compile=False --pos_enc={POS_ENC} --warmup_iters={PRETRAIN_ITERS // 20} \
  --block_size={BLOCK_SIZE} --batch_size={BATCH_SIZE} --gradient_accumulation_steps={GRAD_ACCUM} \
  --n_layer={N_LAYER} --n_head={N_HEAD} --n_embd={N_EMBD} --dropout=0.0 \
  --max_iters={PRETRAIN_ITERS} --lr_decay_iters={PRETRAIN_ITERS} \
  --eval_interval=250 --eval_iters=20 --log_interval=10

Overriding: dataset = codeparrot
Overriding: out_dir = out-code-3000it-5000f
Overriding: device = cpu
Overriding: compile = False
Overriding: block_size = 256
Overriding: batch_size = 12
Overriding: gradient_accumulation_steps = 1
Overriding: n_layer = 4
Overriding: n_head = 4
Overriding: n_embd = 128
Overriding: dropout = 0.0
Overriding: max_iters = 3000
Overriding: lr_decay_iters = 3000
Overriding: eval_interval = 250
Overriding: eval_iters = 20
Overriding: log_interval = 10
tokens per iteration will be: 3,072
Initializing a new model from scratch
defaulting to vocab_size of GPT-2 to 50304 (50257 rounded up for efficiency)
number of parameters: 7.23M
num decayed parameter tensors: 18, with 7,258,112 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters


/home/oliver/Documents/Exchange/deep-learning/project/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
/home/oliver/anaconda3/envs/nanogpt/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


using fused AdamW: False
step 0: train loss 10.5769, val loss 10.5675
iter 0: loss 10.5711, time 13253.83ms, mfu -100.00%
iter 10: loss 10.5593, time 877.23ms, mfu 0.05%
iter 20: loss 10.3940, time 859.52ms, mfu 0.05%
iter 30: loss 10.0868, time 917.31ms, mfu 0.05%
iter 40: loss 10.0264, time 821.81ms, mfu 0.05%
iter 50: loss 9.9885, time 907.48ms, mfu 0.05%
iter 60: loss 9.8624, time 821.34ms, mfu 0.05%
iter 70: loss 9.8857, time 818.83ms, mfu 0.05%
iter 80: loss 9.7312, time 811.03ms, mfu 0.05%
iter 90: loss 9.8461, time 750.89ms, mfu 0.05%
iter 100: loss 9.9421, time 746.83ms, mfu 0.05%
iter 110: loss 9.5424, time 745.57ms, mfu 0.05%
iter 120: loss 9.3199, time 746.99ms, mfu 0.05%
iter 130: loss 9.4047, time 740.23ms, mfu 0.05%
iter 140: loss 9.4644, time 755.59ms, mfu 0.06%
iter 150: loss 9.5648, time 741.18ms, mfu 0.06%
iter 160: loss 9.0901, time 735.02ms, mfu 0.06%
iter 170: loss 8.9608, time 738.62ms, mfu 0.06%
iter 180: loss 8.9674, time 729.80ms, mfu 0.06%
iter 190: loss 8.97

## 3. Build labels from xCodeEval (run once; label 1 = compiles, 0 = compilation error)

In [3]:
import json, random
from huggingface_hub import list_repo_files, hf_hub_download

REPO = "NTU-NLP-sg/xCodeEval"
random.seed(0)

shards = sorted(f for f in list_repo_files(REPO, repo_type="dataset")
                if f.startswith("code_compilation/train/") and f.endswith(".jsonl"))
print(len(shards), "train shards, e.g.", shards[:2])
random.shuffle(shards)

pos, neg = [], []
for shard in shards[:XC_MAX_SHARDS]:
    path = hf_hub_download(REPO, shard, repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            if ex["lang_cluster"] != XC_LANG_CLUSTER or "3" not in ex["lang"]:   # Python 3 / PyPy 3 only
                continue
            code = ex["source_code"]
            if len(code) > 1500:                                    # cheap pre-filter
                continue
            if len(enc.encode_ordinary(code)) > BLOCK_SIZE:        # whole program must fit in context
                continue
            label = 0 if ex["compilation_error"] else 1
            bucket = pos if label == 1 else neg
            if len(bucket) < XC_N_PER_CLASS:
                bucket.append({"code": code, "label": label, "group": ex["src_uid"]})
    print(f"{shard}: pos={len(pos)} neg={len(neg)}")
    if len(pos) >= XC_N_PER_CLASS and len(neg) >= XC_N_PER_CLASS:
        break

json.dump(pos + neg, open(LABEL_FILE, "w", encoding="utf-8"))
print("saved", len(pos) + len(neg), "examples;", len(pos), "compile /", len(neg), "error")

399 train shards, e.g. ['code_compilation/train/train_0000.jsonl', 'code_compilation/train/train_0001.jsonl']


code_compilation/train/train_0150.jsonl: reconstructing file:   0%|          |  0.00B /  102MB            

code_compilation/train/train_0150.jsonl: downloading bytes:           |  0.00B            

code_compilation/train/train_0150.jsonl: pos=1665 neg=19


code_compilation/train/train_0380.jsonl: reconstructing file:   0%|          |  0.00B /  137MB            

...

code_compilation/train/train_0036.jsonl: downloading bytes:           |  0.00B            

code_compilation/train/train_0036.jsonl: pos=5000 neg=260


code_compilation/train/train_0299.jsonl: reconstructing file:   0%|          |  0.00B /  104MB            

code_compilation/train/train_0299.jsonl: downloading bytes:           |  0.00B            

code_compilation/train/train_0299.jsonl: pos=5000 neg=375
saved 5375 examples; 5000 compile / 375 error


## 4. Classifier model (configurable head depth, partial freezing)

In [5]:
import contextlib, torch, torch.nn as nn
from model import GPT, GPTConfig

BLOCK = BLOCK_SIZE
ckpt = torch.load(f"{PRETRAIN_DIR}/ckpt.pt", map_location="cpu", weights_only=False)

def build_gpt(pretrained=True):
    gpt = GPT(GPTConfig(**ckpt["model_args"]))
    if pretrained:
        sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}
        gpt.load_state_dict(sd)
    return gpt

def make_head(n_embd, depth, n_classes=2, hidden=256, p=0.1):
    assert depth in (1, 2, 3)
    layers, d = [nn.Dropout(p)], n_embd
    for _ in range(depth - 1):
        layers += [nn.Linear(d, hidden), nn.GELU(), nn.Dropout(p)]
        d = hidden
    layers.append(nn.Linear(d, n_classes))
    return nn.Sequential(*layers)

class CodeClassifier(nn.Module):
    # n_freeze = number of bottom blocks to freeze (embeddings are frozen whenever n_freeze > 0;
    # n_freeze >= n_layer freezes the final LayerNorm too, i.e. the whole GPT)
    def __init__(self, gpt, head_depth=1, hidden=256, n_classes=2, n_freeze=0):
        super().__init__()
        self.gpt = gpt
        self.n_layer = len(gpt.transformer.h)
        self.n_freeze = min(n_freeze, self.n_layer)
        self.head = make_head(gpt.config.n_embd, head_depth, n_classes, hidden)
        t, k = gpt.transformer, self.n_freeze
        frozen = []
        if k > 0:
            frozen += [t.wte] + ([t.wpe] if "wpe" in t else []) + list(t.h[:k])
        if k >= self.n_layer:
            frozen.append(t.ln_f)
        for mod in frozen:
            for p in mod.parameters():
                p.requires_grad = False

    def forward(self, idx, mask):
        t, k = self.gpt.transformer, self.n_freeze
        no_grad = torch.no_grad() if k > 0 else contextlib.nullcontext()
        with no_grad:                                   # frozen bottom part: no backprop, faster
            x = self.gpt.embed(idx)                     # wte (+ wpe for 'abs'); RoPE acts inside attention
            for block in t.h[:k]:
                x = block(x)
        for block in t.h[k:]:                           # trainable blocks
            x = block(x)
        if k >= self.n_layer:
            with torch.no_grad():
                x = t.ln_f(x)
        else:
            x = t.ln_f(x)
        m = mask.unsqueeze(-1).float()
        pooled = (x * m).sum(1) / m.sum(1)              # mean over real tokens
        return self.head(pooled)

## 5. Data and training helpers

In [6]:
rows_all = json.load(open(LABEL_FILE, encoding="utf-8"))

# split by problem (src_uid) so the same task never appears in both train and test
uids = sorted({r["group"] for r in rows_all})
random.Random(0).shuffle(uids)
test_uids = set(uids[:int(0.2 * len(uids))])
test_rows  = [r for r in rows_all if r["group"] in test_uids]
train_pool = [r for r in rows_all if r["group"] not in test_uids]
random.Random(0).shuffle(train_pool)
print(f"pool: {len(train_pool)} train / {len(test_rows)} test")

def get_train_rows(n_labeled):
    # class-balanced subset of size n_labeled (None = whole pool)
    if n_labeled is None: return list(train_pool)
    k = n_labeled // 2
    pos = [r for r in train_pool if r["label"] == 1][:k]
    neg = [r for r in train_pool if r["label"] == 0][:k]
    return pos + neg

def make_batch(rows):
    ids = [enc.encode_ordinary(r["code"])[:BLOCK] for r in rows]
    L = max(len(i) for i in ids)
    x = torch.zeros(len(ids), L, dtype=torch.long)
    m = torch.zeros(len(ids), L, dtype=torch.long)
    for k, i in enumerate(ids):
        x[k, :len(i)] = torch.tensor(i); m[k, :len(i)] = 1     # right-padded; causal attention => safe
    y = torch.tensor([r["label"] for r in rows])
    return x, m, y

def evaluate(model, rows):
    model.eval(); correct = 0
    with torch.no_grad():
        for i in range(0, len(rows), 32):
            x, m, y = make_batch(rows[i:i+32])
            correct += (model(x, m).argmax(-1) == y).sum().item()
    model.train()
    return correct / len(rows)

def train_classifier(pretrained=True, head_depth=None, n_labeled=None, epochs=None,
                     n_freeze=None, bs=16, verbose=True):
    head_depth = HEAD_DEPTH if head_depth is None else head_depth
    n_labeled  = N_LABELED if n_labeled is None else n_labeled
    epochs     = FT_EPOCHS if epochs is None else epochs
    n_freeze   = N_FREEZE if n_freeze is None else n_freeze

    torch.manual_seed(0); random.seed(0)
    rows = get_train_rows(n_labeled)
    model = CodeClassifier(build_gpt(pretrained), head_depth, HEAD_HIDDEN, n_freeze=n_freeze)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"pretrained={pretrained} depth={head_depth} freeze={n_freeze}/{model.n_layer} "
          f"train={len(rows)} trainable={n_tr/1e6:.2f}M")

    gpt_params = [p for p in model.gpt.parameters() if p.requires_grad]
    groups_ = [{"params": model.head.parameters(), "lr": 1e-3}]
    if gpt_params:
        groups_.append({"params": gpt_params, "lr": 1e-4})
    opt = torch.optim.AdamW(groups_, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()

    accs = []
    model.train()
    for epoch in range(epochs):
        random.shuffle(rows)
        for step, i in enumerate(range(0, len(rows), bs)):
            x, m, y = make_batch(rows[i:i+bs])
            loss = loss_fn(model(x, m), y)
            opt.zero_grad(); loss.backward(); opt.step()
            if verbose and step % 50 == 0:
                print(f"  epoch {epoch} step {step} loss {loss.item():.3f}")
        accs.append(evaluate(model, test_rows))
        print(f"epoch {epoch}: test acc {accs[-1]:.3f}")
    return model, accs

pool: 4116 train / 1259 test


## 6. Single runs

In [7]:
model_pre, accs_pre = train_classifier(pretrained=True)                       # uses the config above
model_scratch, accs_scratch = train_classifier(pretrained=False, n_freeze=0)  # random-init baseline

number of parameters: 7.23M
pretrained=True depth=1 freeze=0/4 train=2637 trainable=7.26M
  epoch 0 step 0 loss 1.179
  epoch 0 step 50 loss 0.350
  epoch 0 step 100 loss 0.062
  epoch 0 step 150 loss 0.438
epoch 0: test acc 0.892
  epoch 1 step 0 loss 0.383
  epoch 1 step 50 loss 0.643
  epoch 1 step 100 loss 0.143
  epoch 1 step 150 loss 0.300
epoch 1: test acc 0.889
  epoch 2 step 0 loss 0.140
  epoch 2 step 50 loss 0.162
  epoch 2 step 100 loss 0.120
  epoch 2 step 150 loss 0.258
epoch 2: test acc 0.893
  epoch 3 step 0 loss 0.046
  epoch 3 step 50 loss 0.435
  epoch 3 step 100 loss 0.020
  epoch 3 step 150 loss 0.131
epoch 3: test acc 0.886
  epoch 4 step 0 loss 0.346
  epoch 4 step 50 loss 0.085
  epoch 4 step 100 loss 0.200
  epoch 4 step 150 loss 0.257
epoch 4: test acc 0.859
  epoch 5 step 0 loss 0.064
  epoch 5 step 50 loss 0.001
  epoch 5 step 100 loss 0.100
  epoch 5 step 150 loss 0.084
epoch 5: test acc 0.878
  epoch 6 step 0 loss 0.010
  epoch 6 step 50 loss 0.019
  epoch

## 7. Experiment grid: freezing x head depth x labelled-data size

In [8]:
import pandas as pd

FREEZE_SETTINGS = {"finetune (0)": 0, "half frozen": N_LAYER // 2, "fully frozen": N_LAYER}
results = []
for name, k in FREEZE_SETTINGS.items():
    for depth in (1, 2, 3):
        for n in (500, 1000, 2500, 5000):
            _, a = train_classifier(pretrained=True, n_freeze=k, head_depth=depth,
                                    n_labeled=n, epochs=5, verbose=False)
            results.append(dict(mode=name, depth=depth, n_labeled=n, final_acc=a[-1]))
for depth in (1, 2, 3):                                                       # scratch baseline
    for n in (500, 1000, 2500, 5000):
        _, a = train_classifier(pretrained=False, n_freeze=0, head_depth=depth,
                                n_labeled=n, epochs=5, verbose=False)
        results.append(dict(mode="scratch", depth=depth, n_labeled=n, final_acc=a[-1]))

df = pd.DataFrame(results)
print(df.pivot_table(index=["mode", "depth"], columns="n_labeled", values="final_acc").round(3))

number of parameters: 7.23M
pretrained=True depth=1 freeze=0/4 train=487 trainable=7.26M
epoch 0: test acc 0.665
epoch 1: test acc 0.774
epoch 2: test acc 0.637
epoch 3: test acc 0.503
epoch 4: test acc 0.622
number of parameters: 7.23M
pretrained=True depth=1 freeze=0/4 train=737 trainable=7.26M
epoch 0: test acc 0.848
epoch 1: test acc 0.873
epoch 2: test acc 0.844
epoch 3: test acc 0.824
epoch 4: test acc 0.753
number of parameters: 7.23M
pretrained=True depth=1 freeze=0/4 train=1487 trainable=7.26M
epoch 0: test acc 0.891
epoch 1: test acc 0.891
epoch 2: test acc 0.867
epoch 3: test acc 0.888
epoch 4: test acc 0.873
number of parameters: 7.23M
pretrained=True depth=1 freeze=0/4 train=2737 trainable=7.26M
epoch 0: test acc 0.892
epoch 1: test acc 0.886
epoch 2: test acc 0.845
epoch 3: test acc 0.768
epoch 4: test acc 0.874
number of parameters: 7.23M
pretrained=True depth=2 freeze=0/4 train=487 trainable=7.29M
epoch 0: test acc 0.754
epoch 1: test acc 0.705
epoch 2: test acc 0.711
e